<a href="https://colab.research.google.com/github/jamshid719/computer_vision/blob/main/menu_detector_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
print('Menu Detector!')

Menu Detector!


In [11]:
# ------------------------------
# Import Libraries
# ------------------------------
from google.colab import drive
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision.models import mobilenet_v2

from PIL import Image, UnidentifiedImageError
from torch.utils.data import Dataset, DataLoader
import os
import numpy as np



In [3]:

drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# Define Dataset Path
DATASET_PATH = '/content/drive/MyDrive/food101_dataset'
print('Dataset_path:', DATASET_PATH)

# uzimizning Classimizni yaratamiz
CUSTOM_CLASS_MAPPING = {
    "hamburger": "hamburger",
    "hot_dog": "hot_dog",
    "chocolate_cake": "dessert",  # label grouping | class consolidation
    "cheesecake": "dessert",      # label grouping | class consolidation
    "kebab": "kebab",
    "pilaf": "pilaf"
}


CLASSES = ['hamburger', 'hot_dog', 'dessert', 'kebab', 'pilaf']
CLASS_TO_IDX = {cls: i for i, cls in enumerate(CLASSES)} #Bu class nomlarini raqamga aylantirib beradigan dictionary yaratadi.
                                                         #Yani model rasm classini matn bilan emas, raqam bilan urganadi.
NUM_CLASSES = len(CLASSES)

print(NUM_CLASSES)
print(CLASS_TO_IDX)

# dataset ni bir xil kurinishda keltirish
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                          std=[0.229, 0.224, 0.225])
])

# Compose — bir nechta transformatsiyalarni ketma-ket bajaradigan funksiya.

# .Resize => rasmning size ni 224x224 pikselga uzgartiradi.

# .ToTensor() — bu eng muhim qadam, ikkita ishni birdan qiladi:
# Rasm formatini H, W, C (Height, Width, Channel) dan C, H, W ga uzgartiradi,
# chunki PyTorch kanal (channel)ni 1- uringa quyishni talab qiladi.
# Piksel qiymatlarini 0–255 oralig'idan 0.0–1.0 oralig'iga utkazadi (255 ga bulish orqali).

# .Normalize => Bu yerdagi mean va std qiymatlari — ImageNet datasetida millionlab rasmlardan hisoblangan
# o'rtacha va standart og'ish qiymatlari (RGB uch kanal uchun alohida-alohida). Bu sonlar de-fakto standart hisoblanadi,
# chunki ko'pchilik pretrained modellar shu qiymatlar bilan o'rgatilgan. pixel = (pixel - mean) / std


Dataset_path: /content/drive/MyDrive/food101_dataset
5
{'hamburger': 0, 'hot_dog': 1, 'dessert': 2, 'kebab': 3, 'pilaf': 4}


In [5]:
# ------------------------
# Custom Dataset Class - kuprik vazifasini bajaradi
# ------------------------

#Bu class rasm fayllarini diskdan o'qib, PyTorch modeliga uzatishga tayyorlaydi.
#Dataset'dan meros oladi (inherit), shuning uchun PyTorch'ning DataLoader'i bilan ishlaydi.
class FoodDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        print('images_length', len(self.images))
        return len(self.images)

# 1ta elementni olish (eng muhim qism):
    def __getitem__(self, idx):
        img_path = self.images[idx] # idx-raqamli rasm yulini ol
        print('image_path', img_path) # uning label'ini ol
        label = self.labels[idx]
        print('label', label)
        try:
            image = Image.open(img_path).convert('RGB') # rasmni och, RGB ga utkaz
        except (UnidentifiedImageError, OSError):
            print(f"Skipping broken image: {img_path}")
            return self.__getitem__((idx + 1) % len(self.images))  # keyingi rasmni ol
        if self.transform:
            image = self.transform(image) # transform qilish
        return image, label

In [6]:
# ------------------------
# Gather and Split Data - Bu kod rasmlarni yigib, training va validation qismlariga ajratadigan qism
# ------------------------
all_images = []
for original_class, mapped_class in CUSTOM_CLASS_MAPPING.items():# original_class — papka nomi (hamburger)
    class_path = os.path.join(DATASET_PATH, original_class) # /content/drive/MyDrive/food101_dataset/hamburger
    print('class_path:', class_path)
    if not os.path.exists(class_path):
        print(f"Warning: {class_path} not found")
        continue
    for img in os.listdir(class_path): # img - papkadagi har bir fayl
        if img.endswith(('.jpg', '.jpeg', '.png')):
            full_path = os.path.join(class_path, img) #/content/drive/MyDrive/food101_dataset/hamburger/100057.jpg
            all_images.append((full_path, CLASS_TO_IDX[mapped_class]))

np.random.shuffle(all_images) # Datalarni random qilib aralashtiramiz.(training ga faqat 1-sinflar(hamma hamburger), validation'ga esa oxirgi sinflar tushib qolmasligi un)
split = int(0.8 * len(all_images)) # 80% nuqtasini hisoblaydi
train_data = all_images[:split] # 80%
val_data = all_images[split:] # 20%

train_images, train_labels = zip(*train_data)
val_images, val_labels = zip(*val_data)

#print('all_images:', all_images)

# Dataset yaratish va sinash:
dataset = FoodDataset(train_images, train_labels) # oldingi klassdan obyekt
print(len(dataset)) # nechta rasm bor
img, lbl = dataset[0] # 1-rasmni sinab kurish va hammasi tugri ishlayotganini tekshiradi.



class_path: /content/drive/MyDrive/food101_dataset/hamburger
class_path: /content/drive/MyDrive/food101_dataset/hot_dog
class_path: /content/drive/MyDrive/food101_dataset/chocolate_cake
class_path: /content/drive/MyDrive/food101_dataset/cheesecake
class_path: /content/drive/MyDrive/food101_dataset/kebab
class_path: /content/drive/MyDrive/food101_dataset/pilaf
images_length 3234
3234
image_path /content/drive/MyDrive/food101_dataset/hamburger/564651.jpg
label 0


In [7]:
train_dataset = FoodDataset(train_images, train_labels, transform=transform)
val_dataset = FoodDataset(val_images, val_labels, transform=transform)

In [8]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2) #thread | parallel loading for speed
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

# DataLoader => Dataset'dan rasmlarni tuplamlar (batch) bulib modelга uzatadi, yani bittalab emas, guruh bn.

#batch_size=32 => bir vaqtda modelga 32 ta rasm beradi. Hammasini birdan bermaydi(xotira yetmaydi).
# bizda 3278 rasmni: 3278 ÷ 32 ≈ 103 ta batch.

#shuffle=True (train uchun) — har epochda rasmlarni aralashtiradi.
#Bu modelning rasmlar tartibini yodlab olishini oldini oladi, yaxshiroq o'rganadi.

#shuffle=False (val uchun) — validation'da aralashtirish shart emas,
#chunki bu yerda model o'rganmaydi, faqat sinaladi. Tartib muhim emas.

#num_workers=2 — rasmlarni 2 ta parallel jarayon (thread) bilan yuklaydi.
#Bu tezlashtiradi — model bir batch'da ishlayotganda, ishchilar kngi batch'ni tayyorlab turadi.

#Umumiy tasvir
#FoodDataset (rasmlarni o'qiydi)
#        ↓
#DataLoader (32 talik batch'larga bo'ladi, aralashtiradi, parallel yuklaydi)
#       ↓
#Model (batch'larni oladi va o'rganadi)

images_length 3234
images_length 3234


In [9]:
# pretrained model - Bu kod tayyor (pretrained) modelni olib, uz modelizga moslashtirish(transfer learning).

model = mobilenet_v2(weights="IMAGENET1K_V1") # pretrained model | ligthweight | CNN | 1000 class | million
model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES) # fine-tuning | backbone | model layer freeze

# mobilenet_v2 — bu allaqachon tayyor uqitilgan CNN (neyron tarmoq) modeli(ishni noldan bowlamaymiz).

# weights="IMAGENET1K_V1" — bu model ImageNet degan ulkan datasetda uqitilgan:
# <1000 ta sinf (it, mushuk, mashina, samolyot va h.k.)
# million+ rasm bilan o'qitilgan

#NUM_CLASSES uzimizni class ni quyib ketamiz

# layer freeze — bu model qatlamlarini uqitish paytida uzgartirmaslik degani,
# yani training paytida ularning ichki qiymatlari (weights) yangilanmaydi, muzlatadi.

#Umumiy Mantiq
# 1. Tayyor, aqlli modelni ol (1000 sinfni biladi)
#         ↓
# 2. Uning "miyasi"ni (backbone) saqla — rasm tushunish qobiliyati
#         ↓
# 3. Faqat oxirgi qaror qatlamini almashtir (1000 → 5)
#         ↓
# 4. Endi model sizning 5 ta ovqatingizni o'rganishga tayyor


Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 222MB/s]


In [10]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device', device)
model = model.to(device)

device cuda


In [13]:
criterion = nn.CrossEntropyLoss() # Loss Function | '70%' burger, '30%' pilaf
optimizer = optim.Adam(model.parameters(), lr=0.001) # weight
torch.backends.cudnn.benchmark = True # Benchmark Setting | Trick | 10%-20%

# CrossEntropyLoss — multi-class classification un eng kup ishlatiladigan loss. Bizda 5 ta sinf bor.

# Adam — eng mashhur va samarali optimizer. Aqlli tarzda o'rganish tezligini moslashtiradi.
# model.parameters() — modelning barcha ourgatiladigan og'irliklari. "Mana shularni sozla" deydi.
# lr=0.001 (learning rate) — eng muhim parametr. Model har qadamda og'irliklarni qanchalik katta uzgartirishini belgilaydi

#benchmark = True — birinchi marta ishlaganda, cuDNN turli algoritmlarni sinab ko'radi va eng tezini tanlaydi.
#Kn shu tez usulni ishlatadi.